# Fine-tuning do Assistente Médico — Tech Challenge Fase 3

Este notebook realiza o fine-tuning de um LLM (Llama 3 8B) com dados médicos
internos sintéticos do hospital, usando **Unsloth + QLoRA** (4-bit), que
roda dentro dos limites de uma GPU gratuita do Google Colab (T4).

**Fluxo deste notebook:**
1. Instalar dependências (Unsloth, bitsandbytes, TRL, PEFT)
2. Carregar o dataset de fine-tuning (protocolos + FAQs + modelos de documento, já anonimizados)
3. Carregar o Llama 3 8B em 4-bit e aplicar LoRA
4. Treinar (SFT) com o dataset
5. Avaliar qualitativamente o modelo
6. Exportar o adapter LoRA para o Hugging Face Hub
7. Gerar o `Modelfile` para importar o modelo fine-tunado no Ollama local

> **Pré-requisito:** Runtime > Change runtime type > GPU (T4).


In [ ]:
# 1) Instalação
# Deixamos o Unsloth escolher as versões compatíveis de torch/transformers/
# peft/trl/bitsandbytes (ele mantém isso atualizado internamente). Forçar uma
# versão manual do trl aqui (como fizemos antes) causa erros de assinatura
# como "unexpected keyword argument 'tokenizer'/'processing_class'", porque
# mistura uma versão do trl com peft/transformers de outra geração.
!pip install --quiet -U unsloth

# Se o Colab já tinha uma versão antiga de algum pacote pré-instalada e algo
# ainda ficar inconsistente após esta célula, use Ambiente de execução >
# Reiniciar sessão e rode as células novamente do início.


## 2) Carregar o dataset de fine-tuning

Opções:
- **(a)** Fazer upload do arquivo `finetuning_dataset.jsonl` gerado localmente por `src/data_prep/generate_synthetic_data.py`.
- **(b)** Gerar o dataset diretamente aqui a partir dos JSONs sintéticos do repositório (clonado abaixo).

In [ ]:
# Clona o repositório para ter acesso aos dados sintéticos e ao script gerador
import os

REPO_URL = "https://github.com/Misadri2/medical-assistant-ai.git"
REPO_DIR = "/content/medical-assistant-ai"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!python -m src.data_prep.generate_synthetic_data


In [ ]:
import json

DATASET_PATH = "data/processed/finetuning_dataset.jsonl"

examples = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        examples.append(json.loads(line))

print(f"Total de exemplos: {len(examples)}")
examples[0]


### Anonimização

Os dados sintéticos deste desafio já não contêm PII real (são fictícios).
Em um cenário real com dados do hospital, este seria o ponto para rodar
`src.data_prep.anonymize` com o backend Presidio **antes** de qualquer
exemplo entrar no dataset de treino. Deixamos a chamada abaixo comentada
como referência de pipeline completo.

In [ ]:
# from src.data_prep.anonymize import get_anonymizer
# anonymizer = get_anonymizer(backend="presidio")
# examples = [
#     {**ex, "output": anonymizer.anonymize(ex["output"])}
#     for ex in examples
# ]


## 3) Formatar o dataset no template de instrução do Llama 3

In [ ]:
from datasets import Dataset

ALPACA_PROMPT = """Abaixo está uma instrução que descreve uma tarefa clínica, combinada com um contexto de entrada. Escreva uma resposta que complete adequadamente a solicitação, baseando-se apenas nos protocolos internos do hospital.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = None  # definido após carregar o tokenizer, na célula seguinte

def formatting_prompts_func(examples_batch):
    instructions = examples_batch["instruction"]
    inputs = examples_batch["input"]
    outputs = examples_batch["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = ALPACA_PROMPT.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

raw_dataset = Dataset.from_list(examples)
raw_dataset


## 4) Carregar Llama 3 8B em 4-bit e aplicar LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None          # None = auto-detecção (Colab T4 -> float16)
LOAD_IN_4BIT = True   # necessário para caber na T4 gratuita

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

EOS_TOKEN = tokenizer.eos_token
dataset = raw_dataset.map(formatting_prompts_func, batched=True)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 5) Treinamento (SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig
import inspect

# A API do trl mudou várias vezes nos últimos anos (TrainingArguments ->
# SFTConfig; tokenizer -> processing_class; e, em versões mais recentes,
# nem tokenizer nem processing_class são aceitos diretamente por SFTTrainer,
# que passa a inferir isso a partir do `model`). Em vez de descobrir na mão
# qual é a combinação certa para a versão instalada neste Colab, o bloco
# abaixo tenta as combinações conhecidas, da mais recente para a mais
# antiga, até uma funcionar.

def _build_sftconfig():
    return SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,           # dataset pequeno (sintético) -> mais épocas
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
    )


def _build_training_arguments():
    return TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    )


def _try_build_trainer():
    last_error = None

    # Tentativa 1: SFTConfig + processing_class (trl mais recente, ~0.12+)
    # Tentativa 2: SFTConfig + tokenizer (trl intermediário)
    # Tentativa 3: SFTConfig sem tokenizer/processing_class (trl novíssimo,
    #              infere a partir do `model`)
    for tokenizer_kwarg in ("processing_class", "tokenizer", None):
        try:
            args = _build_sftconfig()
            trainer_kwargs = dict(model=model, train_dataset=dataset, args=args)
            if tokenizer_kwarg:
                trainer_kwargs[tokenizer_kwarg] = tokenizer
            trainer = SFTTrainer(**trainer_kwargs)
            print(f"SFTTrainer criado com sucesso (SFTConfig, tokenizer_kwarg={tokenizer_kwarg!r}).")
            return trainer
        except TypeError as e:
            last_error = e
            print(f"Tentativa com SFTConfig (tokenizer_kwarg={tokenizer_kwarg!r}) falhou: {e}")

    # Tentativa 4: TrainingArguments + tokenizer, com dataset_text_field/
    #              max_seq_length/packing como kwargs diretos do SFTTrainer
    #              (formato usado por versões antigas do trl, < 0.12)
    try:
        args = _build_training_arguments()
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LENGTH,
            dataset_num_proc=2,
            packing=False,
            args=args,
        )
        print("SFTTrainer criado com sucesso (TrainingArguments, formato antigo).")
        return trainer
    except TypeError as e:
        last_error = e
        print(f"Tentativa com TrainingArguments (formato antigo) falhou: {e}")

    raise last_error


trainer = _try_build_trainer()
trainer_stats = trainer.train()


## 6) Avaliação qualitativa

Requisito do desafio: "Avaliação do modelo e análise dos resultados".
Testamos aqui com perguntas do próprio conjunto de FAQs (avaliação de
memorização/aderência ao protocolo) e com uma pergunta fora do dataset
(avaliação de generalização).

In [ ]:
FastLanguageModel.for_inference(model)

def perguntar(instruction, input_text=""):
    prompt = ALPACA_PROMPT.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Pergunta presente no dataset de treino
print(perguntar("Qual escore uso para decidir profilaxia de TEV em paciente internado?"))


In [ ]:
# Pergunta NÃO presente no dataset (avalia generalização)
print(perguntar("Um paciente chega com dor torácica e supra de ST no ECG. Qual o próximo passo?"))


### Métricas quantitativas simples

Para uma avaliação mais objetiva do relatório técnico, comparamos a
similaridade (ROUGE-L) entre a resposta gerada e a resposta de referência
do protocolo, para um pequeno conjunto de perguntas de validação
(hold-out, não usadas no treino).

In [ ]:
!pip install --quiet rouge-score

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

validation_pairs = [
    (
        "Quando devo acionar a linha de cateterismo em dor torácica?",
        "Sempre que houver supradesnivelamento de ST no ECG realizado até 10 minutos da chegada, com meta de porta-balão de até 90 minutos.",
    ),
    (
        "A partir de qual potássio sérico posso iniciar insulina na cetoacidose diabética?",
        "Somente após o potássio sérico estar igual ou acima de 3,3 mEq/L.",
    ),
]

scores = []
for pergunta, referencia in validation_pairs:
    resposta = perguntar(pergunta)
    score = scorer.score(referencia, resposta)["rougeL"].fmeasure
    scores.append(score)
    print(f"Pergunta: {pergunta}\nScore ROUGE-L: {score:.2f}\n")

print(f"ROUGE-L médio: {sum(scores)/len(scores):.2f}")


## 7) Exportar o adapter LoRA para o Hugging Face Hub

Isso permite baixar o adapter treinado de qualquer máquina (inclusive a
sua local, ao rodar `src/llm/inference.py` com o backend Ollama) sem
precisar manter o Colab aberto.

In [ ]:
from huggingface_hub import login

# Gere um token em https://huggingface.co/settings/tokens (permissão de escrita)
login()  # cola o token quando solicitado

HF_REPO_ID = "Misadri1984/medassist-lora-adapter"

model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)


## 8) Gerar o `Modelfile` para servir o modelo no Ollama local

Depois de baixar o adapter (ou mesclá-lo ao modelo base — recomendado
para servir via Ollama), gere um `Modelfile` e importe localmente:

```bash
ollama create medassist-llama3-8b -f Modelfile
```

Em seguida, defina `LLM_BACKEND=ollama` no `.env` do projeto local para
que `src/llm/inference.py` passe a usar o modelo fine-tunado real em vez
do `MockLLM`.

In [ ]:
# Mescla o adapter LoRA ao modelo base para exportação em formato GGUF (compatível com Ollama)
model.save_pretrained_gguf("medassist_gguf", tokenizer, quantization_method="q4_k_m")


In [ ]:
modelfile_content = '''FROM ./medassist_gguf/unsloth.Q4_K_M.gguf

TEMPLATE \"\"\"### Instrução:
{{ .Prompt }}

### Resposta:
\"\"\"

PARAMETER stop \"### Instrução:\"
PARAMETER temperature 0.3
'''

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile gerado. Baixe a pasta 'medassist_gguf' e o 'Modelfile' e rode localmente:")
print("  ollama create medassist-llama3-8b -f Modelfile")


## Resumo para o relatório técnico

Preencha `docs/relatorio_tecnico.md` com:
- Hiperparâmetros usados (r, alpha, learning rate, épocas — ver célula de treino acima)
- Perda de treino final (`trainer_stats`)
- Exemplos de perguntas/respostas do passo 6
- Scores ROUGE-L do passo de avaliação quantitativa
- Link do repositório no Hugging Face Hub com o adapter
